## F5-TTS (Espanol) - Voice Cloning Batch

Replica el flujo de `QwenTTS_Voice_Cloning.ipynb` usando **F5-TTS** con el checkpoint fine-tuneado a espanol [`jpgallegoar/F5-Spanish`](https://huggingface.co/jpgallegoar/F5-Spanish) (218 h, acentos de LatAm y Espana; arquitectura `F5TTS_Base`).

- Mismas rutas de entrada (`Small_variants/...`) y mismas oraciones objetivo.
- Salida en `TTS_output_F5TTS/...`.
- F5-TTS **necesita la transcripcion del audio de referencia** (`ref_text`), por eso se mantiene el paso de Whisper, igual que en el notebook de Qwen.

In [ ]:
# Verificar GPU / CUDA
import torch

if torch.cuda.is_available():
    print("CUDA disponible. Detalles de la GPU:")
    print(f"  Numero de GPUs: {torch.cuda.device_count()}")
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    print(f"  CUDA: {torch.version.cuda}")
    props = torch.cuda.get_device_properties(0)
    print(f"  Memoria total: {props.total_memory / (1024**3):.2f} GB")
else:
    print("CUDA no disponible. Active el runtime con GPU (T4) en Colab.")

In [ ]:
# Instalar F5-TTS y Whisper
!pip install -q f5-tts openai-whisper

> Si `pip` reinstala `torch`/`numpy`, reinicie el entorno (*Runtime -> Restart session*) y ejecute desde la celda de imports, sin reinstalar.

In [ ]:
import os
import whisper
from huggingface_hub import hf_hub_download
from f5_tts.api import F5TTS

device = "cuda" if torch.cuda.is_available() else "cpu"

# 1) Whisper para transcribir el audio de referencia (ref_text), igual que en Qwen.
#    Si hay OOM en la T4, baje a "medium".
print("Cargando Whisper (large)...")
whisper_model = whisper.load_model("large").to(device)

# 2) F5-TTS con el checkpoint en espanol (arquitectura F5TTS_Base).
print("Descargando checkpoint F5-Spanish...")
ckpt_file  = hf_hub_download("jpgallegoar/F5-Spanish", "model_1200000.safetensors")
vocab_file = hf_hub_download("jpgallegoar/F5-Spanish", "vocab.txt")

print("Cargando F5-TTS...")
f5tts = F5TTS(
    model="F5TTS_Base",      # arquitectura del fine-tune en espanol
    ckpt_file=ckpt_file,
    vocab_file=vocab_file,
    device=device,
)

In [ ]:
sentences = [
    "¿Sabías que el cuerpo humano posee sensores? ¡Sí! ¡Como los robots!",
    "Esos sensores se encargan de capturar la energía que nos rodea.",
    "La transforman y la transportan como sensaciones que llegan al cerebro.",
    "Los sensores más conocidos detectan formas y colores.",
    "También detectan sonidos, olores y gustos.",
    "Además producen muchas sensaciones en la piel, como frío y calor.",
    "¿Hay otros sensores? ¡Claro que sí!",
    "Muchos más son menos conocidos, pero también importantes.",
    "Algunos sensores nos mantienen en equilibrio. Otros actúan cuando comemos algo picante.",
    "Cuando las sensaciones llegan al cerebro, se encuentran con conocimientos guardados en la memoria.",
    "Esos conocimientos se acumulan durante años .Entonces se produce la percepción.",
    "La percepción nos dice qué significa la sensación. También nos dice si hay un mensaje.",
    "Desde pequeños aprendemos los sonidos de nuestra lengua.",
    "Si oímos una palabra que nunca escuchamos, tratamos de acercarla a una conocida.",
    "Si eso falla, podemos decir: ¡no la entendí!"
]

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### Generacion - una variante por corrida

Igual que en el notebook de Qwen: procese una variante cambiando el ultimo segmento de la ruta (`NoDenoising_DNSMOS_2-7` -> la que quiera). La salida solo cambia el nombre del TTS en la carpeta raiz (`TTS_output_F5TTS`).

In [ ]:
folder_variants = "/content/drive/MyDrive/Tesis experiments/Small_variants/NoDenoising_DNSMOS_2-7"
output_folder   = "/content/drive/MyDrive/Tesis experiments/TTS_output_F5TTS/NoDenoising_DNSMOS_2-7"
count_sentence = 0

os.makedirs(output_folder, exist_ok=True)
audio_files = os.listdir(folder_variants)
for audio in audio_files:
    # Oracion objetivo (round-robin), mismo criterio que en Qwen
    infer_sentence = sentences[count_sentence % len(sentences)]
    count_sentence += 1
    ref_audio_path = os.path.join(folder_variants, audio)

    # F5-TTS requiere ref_text -> transcribimos la referencia con Whisper
    transcription = whisper_model.transcribe(ref_audio_path)
    ref_text = transcription["text"].strip()

    # Inferencia: F5 escribe el wav directamente via file_wave
    name_wav = audio.split(".")[0] + ".wav"
    output_path = os.path.join(output_folder, name_wav)
    f5tts.infer(
        ref_file=ref_audio_path,
        ref_text=ref_text,
        gen_text=infer_sentence,
        file_wave=output_path,
        remove_silence=False,
    )

### (Opcional) Barrer todas las variantes
Procesa de una sola vez todas las subcarpetas de `Small_variants/`.

In [ ]:
# OPCIONAL: barrer TODAS las variantes de Small_variants
base_variants = "/content/drive/MyDrive/Tesis experiments/Small_variants"
base_output   = "/content/drive/MyDrive/Tesis experiments/TTS_output_F5TTS"

for config in sorted(os.listdir(base_variants)):
    in_dir = os.path.join(base_variants, config)
    if not os.path.isdir(in_dir):
        continue
    out_dir = os.path.join(base_output, config)
    os.makedirs(out_dir, exist_ok=True)

    count_sentence = 0
    for audio in sorted(os.listdir(in_dir)):
        infer_sentence = sentences[count_sentence % len(sentences)]
        count_sentence += 1
        ref_audio_path = os.path.join(in_dir, audio)

        transcription = whisper_model.transcribe(ref_audio_path)
        ref_text = transcription["text"].strip()

        name_wav = audio.split(".")[0] + ".wav"
        output_path = os.path.join(out_dir, name_wav)
        f5tts.infer(
            ref_file=ref_audio_path,
            ref_text=ref_text,
            gen_text=infer_sentence,
            file_wave=output_path,
            remove_silence=False,
        )
    print(f"[ok] {config}: {count_sentence} audios")

Una vez generados los audios en la carpeta `TTS_output_*`, evaluelos con **NISQA** (estandar y fine-tune en espanol) y vuelque los MOS en `data_mos.py`, igual que con QwenTTS.